# Team Synergy Experiment (Brave et al., 2019)

이 노트북은 Zotero 논문에 기재된 공간 요인 모형(Spatial Factor Model) 수식을 반영한 `BraveEtAlSynergyModel`을 실험합니다.

In [ ]:
import numpy as np
import pandas as pd

from team_synergy_model import BraveEtAlSynergyModel, generate_sample_data

## 1. 가상 데이터 생성
`generate_sample_data()` 함수를 이용해 15명(타자 10, 투수 5)의 가상 데이터(WAR, 출전 비중 $\kappa$, 투수 여부, 실제 팀 승수)를 생성합니다.

In [ ]:
player_war, kappa, is_pitcher, team_wins = generate_sample_data(n_players=15)

print(f"팀 총 WAR: {np.sum(player_war):.2f}")
print(f"대체 선수 수준 합산(Alpha): 50.00")
print(f"WAR 기반 기대 승수: {np.sum(player_war) + 50:.2f}")
print(f"실제 기록된 팀 승수 (Team Wins): {team_wins}")

## 2. 모형 적합 (Spatial SAR Model)
논문 수식 (4), (7), (8)에 따라 팀 승수 부족분(또는 초과분)을 선수들의 기대 공헌도($\hat{W}$)와 WAR의 차이(잔차, $\hat{\epsilon}$)로 분해한 뒤, 네트워크 행렬 $A$와 공간 자귀회귀 계수 $\rho$를 추정하여 시너지 효과를 계산합니다.

In [ ]:
model = BraveEtAlSynergyModel(alpha=50.0)

results = model.fit(player_war, kappa, is_pitcher, team_wins)

print(f"추정된 공간 자귀회귀 계수 (rho): {results['rho']:.4f}")
print(f"총 팀 시너지 (tcWAR): {results['tcWAR']:.2f} Wins")

## 3. 선수별 Complementarity 분석 (pcWAR)
선수별로 계산된 생산성 잔차와 동료들에게 미치는 순 파급 효과(pcWAR)를 확인합니다.

In [ ]:
results_df = pd.DataFrame({
    'Role': ['Pitcher' if p else 'Hitter' for p in is_pitcher],
    'Base_WAR': player_war,
    'Expected_Wins (W_hat)': results['expected_wins'],
    'Productivity_Residual': results['productivity_residuals'],
    'Fundamental_Shock': results['fundamental_shocks'],
    'pcWAR (Synergy Effect)': results['pcWAR']
})
results_df.round(3)